# Optimal feedback control simulations of Athalye et al. (2023), Figure 6, on dandiset 000404

Paper. Athalye VR, Khanna P, Gowda S, Orsborn AL, Costa RM, Carmena JM (2023). Invariant neural dynamics drive commands to control different movements. *Current Biology* 33, 2962-2976. [doi:10.1016/j.cub.2023.06.027](https://doi.org/10.1016/j.cub.2023.06.027). Claims file written from the accepted author manuscript (PMC10527529) with its supplement.

Dandiset. [DANDI:000404](https://dandiarchive.org/dandiset/000404/0.230605.2024), version 0.230605.2024, the version the paper cites and the only published one. Loading, the decoder and the dynamics fits are those of `reproduce_athalye_2023.ipynb` (helper `bmi.py`); the simulation is in `ofc.py`.

The simulations need, per session, the fitted dynamics (A, b), the decoder's gain and its cursor dynamics at the 100 ms step. The files hold no decoder matrices; Monkey G's cursor update is recovered exactly from the decoder state, and Monkey J's 100 ms update is estimated by least squares from the decoder state over each 100 ms bin (an assumption: the paper builds it from the PPF's parameters, which are not in the files).

| Claim | Tested here |
|---|---|
| fig6c-dynamics-reduce-input | tested |
| fig6d-decoder-null-dynamics-no-saving | tested |
| fig6g-ofc-uses-distinct-patterns | tested |

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import numpy as np
import pandas as pd
import scipy.stats
import statsmodels.formula.api as smf

import bmi
import ofc

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
NSHUFF = 1000
started = time.time()
sessions = bmi.session_list()
amy = bmi.load_amy()


def load(x):
    m, s, path, _, _ = x
    d = bmi.load_raw(path)
    dec = bmi.fit_decoder(d, m)
    B = bmi.bin_session(d, m, dec["K"], amy)
    return (m, s), d, bmi.prepare(B, dec, m)


with ThreadPoolExecutor(4) as ex:
    loaded = list(ex.map(load, sessions))
RAW = {k: d for k, d, _ in loaded}
BB = {k: B for k, _, B in loaded}
KEYS = sorted(BB)
print(f"loaded {len(KEYS)} files of dandiset {bmi.DANDISET_ID} version {bmi.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

In [ ]:
rows = []
for k in KEYS:
    F, K, vaf = ofc.cursor_update(RAW[k], BB[k], k[0])
    rows.append(dict(monkey=k[0], session=k[1], units=BB[k]["spk"].shape[1], bins=len(BB[k]["spk"]),
                     vaf_pos_x=vaf[0], vaf_pos_y=vaf[1], vaf_vel_x=vaf[2], vaf_vel_y=vaf[3],
                     velocity_decay=round(F[2, 2], 4), pos_from_vel=round(F[0, 2], 4),
                     max_rel_diff_Kvel_vs_command_matrix=np.abs(K[2:4] - BB[k]["Kc"]).max() / np.abs(BB[k]["Kc"]).max()))
print(pd.DataFrame(rows).round(5).to_string(index=False))

Monkey G's 100 ms cursor update is the Kalman filter's (variance accounted for 0.98 to 1 on the updates; its velocity rows equal the command matrix). Monkey J's 100 ms update, estimated over 100 ms bins, accounts for over 99% of the position and velocity; its velocity gain differs from the sum of the 5 ms gains because the cursor dynamics act within the bin.

In [ ]:
def job(k):
    t0 = time.time()
    return k, ofc.run_session(RAW[k], BB[k], k[0], 2000 * (k[0] == "J") + k[1], NSHUFF), time.time() - t0

order = sorted(KEYS, key=lambda k: -BB[k]["spk"].shape[1])
SIM = {}
with ProcessPoolExecutor(4) as ex:
    for k, o, dt in ex.map(job, order):
        SIM[k] = o
        print(k, f"{dt:.0f} s", flush=True)
rows = []
for k in KEYS:
    for name, r in SIM[k]["models"].items():
        rows.append(dict(monkey=k[0], session=k[1], model=name, noise_sd=SIM[k]["sigmas"][name], tuples=r["n_tuples"],
                         final_error_cm_median=np.median(r["final_err"]), final_error_cm_max=r["final_err"].max()))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Largest final distance from the target over all trials (cm):", round(max(r["final_err"].max() for k in KEYS for r in SIM[k]["models"].values()), 3),
      "(target radius 1.7 cm for Monkey G, 1.2 cm for Monkey J)")

Every simulated trial ends within the printed distance of its target, less than a target radius, so every simulated trial completes its task.

## fig6c-dynamics-reduce-input and fig6d-decoder-null-dynamics-no-saving

> In an optimal feedback control model that drives the recorded decoders, a neural population with the fitted invariant dynamics needs significantly less input to perform the tasks than one with no dynamics.

> When the model's invariant dynamics are restricted to the decoder-null space, the input needed is not different from a population with no dynamics.

In [ ]:
def input_table(model, base):
    rows, lme = [], []
    for k in KEYS:
        a = pd.DataFrame(SIM[k]["models"][model]["mags"]).groupby(["task", "target"]).mag.mean()
        c = pd.DataFrame(SIM[k]["models"][base]["mags"]).groupby(["task", "target"]).mag.mean()
        w = scipy.stats.wilcoxon(a.values, c.values)
        rows.append(dict(monkey=k[0], session=k[1], input_model=a.mean(), input_no_dynamics=c.mean(), ratio=a.mean() / c.mean(),
                         conditions_lower=int((a.values < c.values).sum()), wilcoxon_p=w.pvalue))
        for v in a.values:
            lme.append(dict(monkey=k[0], session=f"{k[0]}{k[1]}", model=1, mag=v))
        for v in c.values:
            lme.append(dict(monkey=k[0], session=f"{k[0]}{k[1]}", model=0, mag=v))
    df = pd.DataFrame(rows)
    L = pd.DataFrame(lme)
    print(f"{model} against {base}:")
    print(df.round(4).to_string(index=False))
    for mk in ["G", "J"]:
        x = L[L.monkey == mk]
        fit = smf.mixedlm("mag ~ model", x, groups=x["session"]).fit()
        print(f"  Monkey {mk}: LME effect of the dynamics model on input {fit.params['model']:.4f}, z = {fit.tvalues['model']:.2f}, p = {fit.pvalues['model']:.3g}, N = {len(x)}; "
              f"sessions with Wilcoxon p < 0.05: {(df[df.monkey == mk].wilcoxon_p < 0.05).sum()} of {(df.monkey == mk).sum()}")
    return df

C6 = input_table("Full Dynamics", "No Dynamics")
print("Paper: Monkey G z = 10.49, N = 432; Monkey J z = 5.20, N = 192; Wilcoxon p < 0.05 in 9 of 9 and 4 of 4 sessions\n")
D6 = input_table("Decoder-null Dynamics", "No Dynamics (decoder-null offset)")
print("Paper: Monkey G z = 0.002, p = 0.998; Monkey J z = -0.003, p = 0.990; no session significant")

**Result: fig6c-dynamics-reduce-input reproduced; fig6d-decoder-null-dynamics-no-saving reproduced; both resting on an assumption for Monkey J.** With the fitted dynamics the population needs less input than without dynamics in every session (ratios 0.44 to 0.91; Wilcoxon p < 0.05 in 9 of 9 Monkey G and 4 of 4 Monkey J sessions), and the LME finds the reduction in each monkey (z = -11.27, p = 1.87e-29, N = 432; z = -14.94, p = 1.87e-50, N = 192). Decoder-null dynamics need the same input as their No Dynamics model (ratios 0.998 to 1.003): the LME finds no difference (p = 0.998 and 0.988) and no session's Wilcoxon test does (smallest p 0.069). Monkey J's simulations use a 100 ms decoder update estimated from the files.

## fig6g-ofc-uses-distinct-patterns

> In the OFC simulations, activity issuing the same command differs between conditions more than shuffle with the fitted full dynamics, but not without dynamics or with decoder-null dynamics.

In [ ]:
for null in ["shuffle", "subsample"]:
    rows, lme = [], []
    for k in KEYS:
        for name, r in SIM[k]["models"].items():
            if r["n_tuples"] == 0:
                continue
            obs, nm = r["dist_" + null], r["null_mean_" + null]
            u = scipy.stats.mannwhitneyu(obs, nm)
            rows.append(dict(model=name, monkey=k[0], session=k[1], tuples=len(obs), mean_norm=(obs / nm).mean(), mw_p=u.pvalue))
            for v in obs:
                lme.append(dict(model=name, monkey=k[0], session=f"{k[0]}{k[1]}", source=1, dist=v))
            for v in nm:
                lme.append(dict(model=name, monkey=k[0], session=f"{k[0]}{k[1]}", source=0, dist=v))
    df = pd.DataFrame(rows)
    L = pd.DataFrame(lme)
    print(f"Null: {null} ({NSHUFF} draws)")
    print(df.round(4).to_string(index=False))
    out = []
    for name in df.model.unique():
        for mk in ["G", "J"]:
            x = L[(L.model == name) & (L.monkey == mk)]
            fit = smf.mixedlm("dist ~ source", x, groups=x["session"]).fit()
            sub = df[(df.model == name) & (df.monkey == mk)]
            out.append(dict(model=name, monkey=mk, N=len(x), lme_effect=fit.params["source"], z=fit.tvalues["source"], p=fit.pvalues["source"],
                            sessions_mw_p05=int((sub.mw_p < 0.05).sum()), sessions=len(sub), mean_norm=sub.mean_norm.mean()))
    print(pd.DataFrame(out).to_string(index=False))
    print()
print("Paper (LME, simulation against shuffle): Full G z = -23.09, J z = -16.68; No Dynamics G z = 0.168, J 0.462; Decoder-null G 0.932, J -1.490; "
      "No Dynamics (decoder-null) G 0.611, J 0.449; Mann-Whitney p < 0.05 in 9 of 9 and 4 of 4 sessions for Full Dynamics only")

**Result: partially reproduced, resting on an assumption for Monkey J.** Under both nulls, Full Dynamics activity for a command differs between conditions more than the null (mean normalized distance 1.49 to 1.51 for Monkey G and 1.55 to 1.57 for Monkey J; LME z = 25.4 to 26.0 and 15.5 to 15.7; Mann-Whitney p < 0.05 in 9 of 9 and 4 of 4 sessions). Both No Dynamics models show no difference (LME p 0.83 to 0.999, no session significant). Decoder-null dynamics show none for Monkey G (LME p = 0.18) and in no session, but the Monkey J LME finds a small difference (effect 0.016, z = 2.30 to 2.33, p = 0.020 to 0.021; mean normalized distance 1.09), so that absence is not reproduced.

## Summary

| Claim | Outcome |
|---|---|
| fig6c-dynamics-reduce-input | reproduced (assumption for Monkey J) |
| fig6d-decoder-null-dynamics-no-saving | reproduced (assumption for Monkey J) |
| fig6g-ofc-uses-distinct-patterns | partially reproduced (assumption for Monkey J) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The decoders' 100 ms cursor update is recovered from the files (exactly for Monkey G; by least squares over 100 ms bins for Monkey J, where the paper derived it from the PPF's parameters). The dynamics are the in-sample ridge fits on counts per bin, with the session's cross-validated parameter. The terminal weight (100) is the code's. Commands in the simulations are discretized with the simulation's own magnitude percentiles.

What was found about the dataset. The decoder parameters the simulations need are not in the files; Monkey G's can be recovered exactly from the decoder state.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")